# CIFAR-10: SmallResNet + DS-Compact (SGD recipe) + Static INT8 PTQ

In [ ]:
# Set physical GPU before importing torch.
import os
GPU_INDEX = None
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
if GPU_INDEX is not None:
    os.environ["CUDA_VISIBLE_DEVICES"] = GPU_INDEX

# Limit CPU framework threads; DataLoader workers handle image transforms.
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")

import gc
import time
import json
import random
import math
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

SEED = 42
DATA_ROOT = Path("./data")
BATCH_SIZE = 128
EPOCHS = 150
WARMUP_EPOCHS = 5
LEARNING_RATE = 0.05
WEIGHT_DECAY = 5e-4
MIN_LR = 1e-4
NUM_WORKERS = 4
VAL_WORKERS = 4
USE_AMP = True
CPU_THREADS = 1
CALIBRATION_PER_CLASS = 205
CALIBRATION_BATCH_SIZE = 128
TEST_BATCH_SIZE = 256

# Set to "half" only for a separate normalization ablation run.
NORMALIZATION_MODE = "cifar_stats"  # "cifar_stats" or "half"
CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD = (0.2470, 0.2435, 0.2616)

TAG = f"sgd_additional_{NORMALIZATION_MODE}"
CKPT_DIR = Path(f"checkpoints_{TAG}")
RESULT_DIR = Path(f"results_{TAG}")
INT8_DIR = Path(f"checkpoints_int8_{TAG}")
TS_DIR = Path(f"checkpoints_torchscript_{TAG}")
PTQ_RESULT_DIR = Path(f"results_ptq_{TAG}")
for p in [CKPT_DIR, RESULT_DIR, INT8_DIR, TS_DIR, PTQ_RESULT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type == "cuda":
    torch.backends.cudnn.benchmark = True
    torch.backends.cudnn.deterministic = False
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

torch.set_num_threads(1)
try:
    torch.set_num_interop_threads(1)
except RuntimeError:
    pass

print("PyTorch:", torch.__version__)
print("TorchVision:", __import__("torchvision").__version__)
print("Device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
print("Batch:", BATCH_SIZE, "Epochs:", EPOCHS, "Workers:", NUM_WORKERS)
print("Optimizer: SGD + Nesterov | LR:", LEARNING_RATE, "| Scheduler: 5 warm-up + cosine")
print("Normalization:", NORMALIZATION_MODE)

if NORMALIZATION_MODE == "cifar_stats":
    NORM_MEAN, NORM_STD = CIFAR_MEAN, CIFAR_STD
elif NORMALIZATION_MODE == "half":
    NORM_MEAN, NORM_STD = (0.5, 0.5, 0.5), (0.5, 0.5, 0.5)
else:
    raise ValueError("NORMALIZATION_MODE must be 'cifar_stats' or 'half'.")

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    # Fast convolution algorithms; results are reproducible at the seed/split level,
    # but not guaranteed bitwise deterministic across GPU kernels.
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = (DEVICE.type == "cuda")

set_seed(SEED)


## 1. CIFAR-10 DataLoader

Stratified split: 클래스별 4,500장 train + 500장 validation. seed=42. Augmentation은 train에만 적용

In [ ]:
def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)


def make_stratified_indices(targets, seed=SEED):
    targets = np.asarray(targets)
    rng = np.random.default_rng(seed)
    train_indices, val_indices, train_indices_by_class = [], [], {}
    for class_id in range(10):
        class_indices = np.flatnonzero(targets == class_id)
        rng.shuffle(class_indices)
        train_cls = class_indices[:4500].tolist()
        val_cls = class_indices[4500:].tolist()
        train_indices.extend(train_cls)
        val_indices.extend(val_cls)
        train_indices_by_class[class_id] = train_cls
    return train_indices, val_indices, train_indices_by_class


def make_loaders(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, seed=SEED):
    train_transform = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.ToTensor(),
        transforms.Normalize(NORM_MEAN, NORM_STD),
    ])
    eval_transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(NORM_MEAN, NORM_STD),
    ])

    train_aug = datasets.CIFAR10(
        root=str(DATA_ROOT), train=True, download=True, transform=train_transform
    )
    train_eval = datasets.CIFAR10(
        root=str(DATA_ROOT), train=True, download=True, transform=eval_transform
    )
    test_dataset = datasets.CIFAR10(
        root=str(DATA_ROOT), train=False, download=True, transform=eval_transform
    )

    train_indices, val_indices, class_train_indices = make_stratified_indices(
        train_aug.targets, seed
    )
    train_subset = Subset(train_aug, train_indices)
    val_subset = Subset(train_eval, val_indices)

    generator = torch.Generator().manual_seed(seed)
    train_kwargs = dict(
        batch_size=batch_size,
        num_workers=num_workers,
        pin_memory=(DEVICE.type == "cuda"),
        persistent_workers=(num_workers > 0),
        worker_init_fn=seed_worker,
        generator=generator,
        shuffle=True,
    )
    if num_workers > 0:
        train_kwargs["prefetch_factor"] = 2

    val_workers = min(VAL_WORKERS, num_workers)
    val_kwargs = dict(
        batch_size=batch_size,
        num_workers=val_workers,
        pin_memory=(DEVICE.type == "cuda"),
        persistent_workers=(val_workers > 0),
        worker_init_fn=seed_worker,
        shuffle=False,
    )
    if val_workers > 0:
        val_kwargs["prefetch_factor"] = 2

    train_loader = DataLoader(train_subset, **train_kwargs)
    val_loader = DataLoader(val_subset, **val_kwargs)
    test_loader = DataLoader(
        test_dataset, batch_size=TEST_BATCH_SIZE, shuffle=False, num_workers=0
    )

    # Calibration subset: training indices only, evenly sampled by class.
    cal_indices = []
    for c in range(10):
        cal_indices.extend(class_train_indices[c][:CALIBRATION_PER_CLASS])
    calibration_subset = Subset(train_eval, cal_indices)
    calibration_loader = DataLoader(
        calibration_subset, batch_size=CALIBRATION_BATCH_SIZE,
        shuffle=False, num_workers=0
    )
    return train_loader, val_loader, calibration_loader, test_loader, train_eval.classes

train_loader, val_loader, calibration_loader, test_loader, classes = make_loaders()
print("Train / Val / Calibration / Test:",
      len(train_loader.dataset), len(val_loader.dataset),
      len(calibration_loader.dataset), len(test_loader.dataset))
xb, yb = next(iter(train_loader))
print("Train batch:", tuple(xb.shape), tuple(yb.shape))
del xb, yb


## 2. PTQ-compatible model definitions

In [ ]:
class ConvBNReLU(nn.Sequential):
    def __init__(self, in_channels, out_channels, kernel_size=3, stride=1,
                 padding=None, groups=1):
        if padding is None:
            padding = kernel_size // 2
        super().__init__(
            nn.Conv2d(in_channels, out_channels, kernel_size, stride,
                      padding, groups=groups, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=False),
        )

    def fuse_model(self):
        torch.ao.quantization.fuse_modules(self, ["0", "1", "2"], inplace=True)


class SmallResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(
            in_channels, out_channels, 3, stride=stride, padding=1, bias=False
        )
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu1 = nn.ReLU(inplace=False)
        self.conv2 = nn.Conv2d(
            out_channels, out_channels, 3, stride=1, padding=1, bias=False
        )
        self.bn2 = nn.BatchNorm2d(out_channels)
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels),
            )
        else:
            self.shortcut = nn.Identity()
        self.skip_add = nn.quantized.FloatFunctional()
        self.relu_out = nn.ReLU(inplace=False)

    def forward(self, x):
        identity = self.shortcut(x)
        out = self.relu1(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = self.skip_add.add(out, identity)
        return self.relu_out(out)

    def fuse_model(self):
        torch.ao.quantization.fuse_modules(
            self, ["conv1", "bn1", "relu1"], inplace=True
        )
        torch.ao.quantization.fuse_modules(
            self, ["conv2", "bn2"], inplace=True
        )
        if isinstance(self.shortcut, nn.Sequential):
            torch.ao.quantization.fuse_modules(
                self.shortcut, ["0", "1"], inplace=True
            )


class SmallResNetCIFAR(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.quant = torch.ao.quantization.QuantStub()
        self.dequant = torch.ao.quantization.DeQuantStub()
        self.stem = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=False),
        )
        self.layer1 = SmallResidualBlock(32, 32, stride=1)
        self.layer2 = SmallResidualBlock(32, 64, stride=2)
        self.layer3 = SmallResidualBlock(64, 128, stride=2)
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.quant(x)
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.gap(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)
        return self.dequant(x)

    def fuse_model(self):
        torch.ao.quantization.fuse_modules(
            self.stem, ["0", "1", "2"], inplace=True
        )
        for module in self.modules():
            if isinstance(module, SmallResidualBlock):
                module.fuse_model()


class DepthwiseSeparableBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.depthwise = ConvBNReLU(
            in_channels, in_channels, kernel_size=3,
            stride=stride, padding=1, groups=in_channels
        )
        self.pointwise = ConvBNReLU(
            in_channels, out_channels, kernel_size=1, stride=1, padding=0
        )

    def forward(self, x):
        return self.pointwise(self.depthwise(x))

    def fuse_model(self):
        self.depthwise.fuse_model()
        self.pointwise.fuse_model()


class DSCompactCIFAR(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.quant = torch.ao.quantization.QuantStub()
        self.dequant = torch.ao.quantization.DeQuantStub()
        self.stem = ConvBNReLU(3, 32)
        self.blocks = nn.Sequential(
            DepthwiseSeparableBlock(32, 64, stride=2),
            DepthwiseSeparableBlock(64, 128, stride=2),
            DepthwiseSeparableBlock(128, 128, stride=1),
        )
        self.avgpool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.quant(x)
        x = self.stem(x)
        x = self.blocks(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return self.dequant(x)

    def fuse_model(self):
        self.stem.fuse_model()
        for block in self.blocks:
            block.fuse_model()

MODEL_CLASSES = {
    "small_resnet": SmallResNetCIFAR,
    "ds_compact": DSCompactCIFAR,
}
MODEL_NAMES_TO_TRAIN = list(MODEL_CLASSES.keys())


## 3. Initialization, SGD training, warm-up + cosine

In [ ]:
def initialize_model(model):
    for module in model.modules():
        if isinstance(module, nn.Conv2d):
            nn.init.kaiming_normal_(
                module.weight, mode="fan_out", nonlinearity="relu"
            )
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.BatchNorm2d):
            if module.weight is not None:
                nn.init.ones_(module.weight)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Linear):
            nn.init.kaiming_uniform_(module.weight, a=np.sqrt(5))
            if module.bias is not None:
                nn.init.zeros_(module.bias)
    # Deliberately no zero-init of SmallResidualBlock.bn2:
    # the earlier SmallResNet run did not use this extra stabilization.


def autocast_context():
    if USE_AMP and DEVICE.type == "cuda":
        return torch.autocast(device_type="cuda", dtype=torch.float16)
    from contextlib import nullcontext
    return nullcontext()


def train_one_epoch(model, loader, criterion, optimizer, scaler):
    model.train()
    total_loss = torch.zeros((), device=DEVICE, dtype=torch.float32)
    total_correct = torch.zeros((), device=DEVICE, dtype=torch.long)
    total_count = 0
    for images, labels in loader:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        if DEVICE.type == "cuda":
            images = images.contiguous(memory_format=torch.channels_last)
        optimizer.zero_grad(set_to_none=True)
        with autocast_context():
            logits = model(images)
            loss = criterion(logits, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        n = labels.size(0)
        total_loss += loss.detach().float() * n
        total_correct += (logits.detach().argmax(1) == labels).sum()
        total_count += n
    return (total_loss / total_count).item(), (total_correct.float() / total_count).item()


@torch.inference_mode()
def evaluate_model(model, loader, criterion):
    model.eval()
    total_loss = torch.zeros((), device=DEVICE, dtype=torch.float32)
    total_correct = torch.zeros((), device=DEVICE, dtype=torch.long)
    total_count = 0
    for images, labels in loader:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        if DEVICE.type == "cuda":
            images = images.contiguous(memory_format=torch.channels_last)
        with autocast_context():
            logits = model(images)
            loss = criterion(logits, labels)
        n = labels.size(0)
        total_loss += loss.float() * n
        total_correct += (logits.argmax(1) == labels).sum()
        total_count += n
    return (total_loss / total_count).item(), (total_correct.float() / total_count).item()


def fit_sgd_candidate(model, model_name, train_loader, val_loader, epochs=EPOCHS):
    initialize_model(model)
    model = model.to(DEVICE)
    if DEVICE.type == "cuda":
        model = model.to(memory_format=torch.channels_last)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.SGD(
        model.parameters(), lr=LEARNING_RATE,
        momentum=0.9, nesterov=True, weight_decay=WEIGHT_DECAY
    )
    warmup_epochs = min(WARMUP_EPOCHS, max(0, epochs - 1))
    if warmup_epochs > 0:
        warmup = torch.optim.lr_scheduler.LinearLR(
            optimizer, start_factor=0.2, end_factor=1.0,
            total_iters=warmup_epochs
        )
        cosine = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=epochs - warmup_epochs, eta_min=MIN_LR
        )
        scheduler = torch.optim.lr_scheduler.SequentialLR(
            optimizer, schedulers=[warmup, cosine], milestones=[warmup_epochs]
        )
    else:
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=epochs, eta_min=MIN_LR
        )

    scaler = torch.amp.GradScaler(
        "cuda", enabled=(USE_AMP and DEVICE.type == "cuda")
    )
    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": [], "lr": []}
    best_val_acc, best_val_loss, best_epoch = -1.0, float("inf"), 0
    checkpoint_path = CKPT_DIR / f"{model_name}_best.pth"

    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    start = time.perf_counter()

    for epoch in range(epochs):
        current_lr = optimizer.param_groups[0]["lr"]
        train_loss, train_acc = train_one_epoch(
            model, train_loader, criterion, optimizer, scaler
        )
        val_loss, val_acc = evaluate_model(model, val_loader, criterion)
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)
        history["lr"].append(current_lr)

        is_better = (val_acc > best_val_acc) or (
            np.isclose(val_acc, best_val_acc) and val_loss < best_val_loss
        )
        if is_better:
            best_val_acc, best_val_loss, best_epoch = val_acc, val_loss, epoch + 1
            cpu_state = {k: v.detach().cpu() for k, v in model.state_dict().items()}
            torch.save(cpu_state, checkpoint_path)

        scheduler.step()  # Do not pass epoch; avoids the deprecated epoch argument.
        print(
            f"[{model_name}] Epoch {epoch+1:03d}/{epochs} "
            f"LR={current_lr:.6g} "
            f"Train loss={train_loss:.4f}, acc={train_acc*100:.2f}% | "
            f"Val loss={val_loss:.4f}, acc={val_acc*100:.2f}%"
        )

    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    elapsed = time.perf_counter() - start
    state = torch.load(checkpoint_path, map_location=DEVICE, weights_only=True)
    model.load_state_dict(state)

    with open(RESULT_DIR / f"{model_name}_history.json", "w") as f:
        json.dump(history, f, indent=2)

    summary = {
        "model": model_name,
        "parameters": sum(p.numel() for p in model.parameters() if p.requires_grad),
        "best_epoch": best_epoch,
        "best_val_accuracy_pct": best_val_acc * 100,
        "best_val_loss": best_val_loss,
        "training_minutes": elapsed / 60,
        "checkpoint": str(checkpoint_path),
        "normalization_mode": NORMALIZATION_MODE,
    }
    print("Best checkpoint:", summary)
    return model, history, summary


def save_curves(history, model_name):
    x = np.arange(1, len(history["train_loss"]) + 1)
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
    axes[0].plot(x, history["train_loss"], label="Train")
    axes[0].plot(x, history["val_loss"], label="Validation")
    axes[0].set(title="Loss", xlabel="Epoch", ylabel="CE loss")
    axes[0].legend(); axes[0].grid(alpha=0.25)
    axes[1].plot(x, np.array(history["train_acc"])*100, label="Train")
    axes[1].plot(x, np.array(history["val_acc"])*100, label="Validation")
    axes[1].set(title="Accuracy", xlabel="Epoch", ylabel="Accuracy (%)")
    axes[1].legend(); axes[1].grid(alpha=0.25)
    axes[2].plot(x, history["lr"])
    axes[2].set(title="Learning rate", xlabel="Epoch", ylabel="LR")
    axes[2].grid(alpha=0.25)
    fig.tight_layout()
    fig.savefig(RESULT_DIR / f"{model_name}_curves.png", dpi=160)
    plt.close(fig)


## 4. 사전 점검 및 두 모델 학습

먼저 `EPOCHS=2`로 smoke test를 수행한 뒤 150으로 수행

In [ ]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

# Model shape / MACs smoke test (CPU; MACs is architecture-only).
for name, cls in MODEL_CLASSES.items():
    candidate = cls().cpu().eval()
    with torch.inference_mode():
        out = candidate(torch.zeros(2, 3, 32, 32))
    assert tuple(out.shape) == (2, 10), (name, out.shape)
    print(name, "output:", tuple(out.shape), "parameters:", count_parameters(candidate))
    del candidate

# For a quick pipeline check, set EPOCHS=2 above and run this cell.
training_summaries = []
training_histories = {}

for model_name, model_class in MODEL_CLASSES.items():
    print("\n" + "="*84)
    print(f"Training {model_name}: SGD + Nesterov + Warm-up/Cosine")
    print("="*84)
    set_seed(SEED)
    train_loader, val_loader, calibration_loader, test_loader, classes = make_loaders()
    model = model_class()
    model, history, summary = fit_sgd_candidate(
        model, model_name, train_loader, val_loader, epochs=EPOCHS
    )
    save_curves(history, model_name)
    training_summaries.append(summary)
    training_histories[model_name] = history

    del model, train_loader, val_loader, calibration_loader, test_loader
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

summary_df = pd.DataFrame(training_summaries).sort_values(
    "best_val_accuracy_pct", ascending=False
)
summary_df.to_csv(RESULT_DIR / "training_summary.csv", index=False)
summary_df


## 5. FP32 CPU evaluation + Static INT8 PTQ

Calibration은 train split에서 클래스별 205장씩 추출한 2,050장만 사용한다. Validation/Test는 Calibration에 사용하지 않는다.

In [ ]:
# Build CPU-only calibration and test loaders using the same normalization as training.
# Recreate datasets from the same stratified seed so calibration indices are train-only.
eval_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(NORM_MEAN, NORM_STD),
])
train_eval_dataset = datasets.CIFAR10(
    root=str(DATA_ROOT), train=True, download=True, transform=eval_transform
)
test_dataset = datasets.CIFAR10(
    root=str(DATA_ROOT), train=False, download=True, transform=eval_transform
)
train_indices, val_indices, class_train_indices = make_stratified_indices(
    train_eval_dataset.targets, SEED
)
cal_indices = []
for c in range(10):
    cal_indices.extend(class_train_indices[c][:CALIBRATION_PER_CLASS])
calibration_dataset = Subset(train_eval_dataset, cal_indices)
calibration_loader = DataLoader(
    calibration_dataset, batch_size=CALIBRATION_BATCH_SIZE, shuffle=False, num_workers=0
)
test_loader_cpu = DataLoader(
    test_dataset, batch_size=TEST_BATCH_SIZE, shuffle=False, num_workers=0
)
classes = train_eval_dataset.classes
print("Calibration images:", len(calibration_dataset), "| Test images:", len(test_dataset))

# Select supported x86 quantization backend.
supported = torch.backends.quantized.supported_engines
if "x86" in supported:
    QUANT_BACKEND = "x86"
elif "fbgemm" in supported:
    QUANT_BACKEND = "fbgemm"
else:
    raise RuntimeError(f"No x86 quantization backend available: {supported}")
torch.backends.quantized.engine = QUANT_BACKEND
torch.set_num_threads(CPU_THREADS)
print("Quantization backend:", QUANT_BACKEND)


def load_fp32_model(model_name):
    model = MODEL_CLASSES[model_name]().cpu()
    path = CKPT_DIR / f"{model_name}_best.pth"
    state = torch.load(path, map_location="cpu", weights_only=True)
    model.load_state_dict(state, strict=True)
    return model.eval()


@torch.inference_mode()
def evaluate_cpu(model, loader):
    model.eval()
    correct = 0
    total = 0
    per_class_correct = [0] * 10
    per_class_total = [0] * 10
    for images, labels in loader:
        logits = model(images)
        pred = logits.argmax(1)
        correct += (pred == labels).sum().item()
        total += labels.numel()
        for c in range(10):
            mask = labels == c
            per_class_total[c] += mask.sum().item()
            per_class_correct[c] += ((pred == labels) & mask).sum().item()
    per_class = {
        classes[c]: 100.0 * per_class_correct[c] / max(1, per_class_total[c])
        for c in range(10)
    }
    return correct / total, per_class


def benchmark_latency(model, sample, warmup=40, repeats=250, trials=5):
    model.eval()
    x = sample[:1].contiguous()
    with torch.inference_mode():
        for _ in range(warmup):
            model(x)
        values = []
        for _ in range(trials):
            start = time.perf_counter()
            for _ in range(repeats):
                model(x)
            values.append((time.perf_counter() - start) * 1000 / repeats)
    return float(np.median(values)), float(np.mean(values))


ptq_rows = []
classwise_rows = []

for model_name in MODEL_CLASSES:
    print("\nPTQ:", model_name)
    fp32_model = load_fp32_model(model_name)
    fp_acc, fp_class = evaluate_cpu(fp32_model, test_loader_cpu)

    # Save FP32 TorchScript so size comparison uses the same serialization format.
    example = torch.zeros(1, 3, 32, 32)
    fp_script = torch.jit.freeze(torch.jit.trace(fp32_model, example).eval())
    fp_script_path = TS_DIR / f"{model_name}_fp32.pt"
    torch.jit.save(fp_script, fp_script_path)

    # Prepare a separate copy of the trained FP32 model for eager static PTQ.
    q_model = load_fp32_model(model_name)
    q_model.fuse_model()
    q_model.qconfig = torch.ao.quantization.get_default_qconfig(QUANT_BACKEND)
    prepared = torch.ao.quantization.prepare(q_model, inplace=False).eval()

    calibrated = 0
    with torch.inference_mode():
        for images, _ in calibration_loader:
            prepared(images.cpu())
            calibrated += images.size(0)

    quantized = torch.ao.quantization.convert(prepared, inplace=False).eval()
    with torch.inference_mode():
        qout = quantized(torch.zeros(2, 3, 32, 32))
    assert tuple(qout.shape) == (2, 10), (model_name, qout.shape)

    int8_acc, int8_class = evaluate_cpu(quantized, test_loader_cpu)
    int8_state_path = INT8_DIR / f"{model_name}_int8_state_dict.pth"
    torch.save(quantized.state_dict(), int8_state_path)
    int8_script = torch.jit.freeze(torch.jit.trace(quantized, example).eval())
    int8_script_path = TS_DIR / f"{model_name}_int8.pt"
    torch.jit.save(int8_script, int8_script_path)

    # Measure model-only CPU batch-1 latency with the same input and thread count.
    sample, _ = test_dataset[0]
    sample = sample.unsqueeze(0)
    fp_med, fp_mean = benchmark_latency(fp_script, sample)
    int_med, int_mean = benchmark_latency(int8_script, sample)

    fp_size = fp_script_path.stat().st_size
    int_size = int8_script_path.stat().st_size
    row = {
        "model": model_name,
        "fp32_test_accuracy_pct": fp_acc * 100,
        "int8_test_accuracy_pct": int8_acc * 100,
        "accuracy_drop_pp": (fp_acc - int8_acc) * 100,
        "fp32_torchscript_mib": fp_size / (1024**2),
        "int8_torchscript_mib": int_size / (1024**2),
        "size_reduction_pct": (1 - int_size / fp_size) * 100,
        "compression_ratio": fp_size / int_size,
        "fp32_batch1_median_ms": fp_med,
        "int8_batch1_median_ms": int_med,
        "fp32_batch1_mean_ms": fp_mean,
        "int8_batch1_mean_ms": int_mean,
        "int8_speedup": fp_med / int_med,
        "cpu_threads": torch.get_num_threads(),
        "quant_backend": QUANT_BACKEND,
        "calibration_images": calibrated,
        "normalization_mode": NORMALIZATION_MODE,
    }
    ptq_rows.append(row)
    for class_name in classes:
        classwise_rows.append({
            "model": model_name,
            "class": class_name,
            "fp32_accuracy_pct": fp_class[class_name],
            "int8_accuracy_pct": int8_class[class_name],
            "accuracy_drop_pp": fp_class[class_name] - int8_class[class_name],
        })

    print(
        f"{model_name}: FP32={fp_acc*100:.2f}% | INT8={int8_acc*100:.2f}% | "
        f"drop={(fp_acc-int8_acc)*100:.2f} pp | "
        f"size={fp_size/1024**2:.3f}->{int_size/1024**2:.3f} MiB | "
        f"latency={fp_med:.3f}->{int_med:.3f} ms | speedup={fp_med/int_med:.2f}x"
    )

    del fp32_model, fp_script, q_model, prepared, quantized, int8_script
    gc.collect()

ptq_df = pd.DataFrame(ptq_rows).sort_values(
    "int8_test_accuracy_pct", ascending=False
)
classwise_df = pd.DataFrame(classwise_rows)
ptq_df.to_csv(PTQ_RESULT_DIR / "ptq_comparison.csv", index=False)
classwise_df.to_csv(PTQ_RESULT_DIR / "ptq_classwise_comparison.csv", index=False)
ptq_df


## 6. 저장된 결과

`training_summary.csv`, 학습 곡선 PNG, `ptq_comparison.csv`, `ptq_classwise_comparison.csv`가 생성된다.

In [ ]:
# Save experiment configuration for the report.
config = {
    "seed": SEED,
    "normalization_mode": NORMALIZATION_MODE,
    "normalization_mean": NORM_MEAN,
    "normalization_std": NORM_STD,
    "batch_size": BATCH_SIZE,
    "epochs": EPOCHS,
    "optimizer": "SGD + momentum=0.9 + Nesterov",
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "scheduler": "5-epoch LinearLR warm-up (start_factor=0.2) + CosineAnnealingLR",
    "min_lr": MIN_LR,
    "augmentation": "RandomCrop(32,padding=4) + RandomHorizontalFlip(0.5)",
    "train_val_test": [45000, 5000, 10000],
    "amp": USE_AMP and DEVICE.type == "cuda",
    "num_workers": NUM_WORKERS,
    "quant_backend": QUANT_BACKEND,
    "calibration_images": CALIBRATION_PER_CLASS * 10,
    "cpu_threads": CPU_THREADS,
}
with open(RESULT_DIR / "experiment_config.json", "w") as f:
    json.dump(config, f, indent=2)

print("Training summary:")
print((RESULT_DIR / "training_summary.csv").resolve())
print("PTQ comparison:")
print((PTQ_RESULT_DIR / "ptq_comparison.csv").resolve())
print("Saved files:")
for folder in [CKPT_DIR, RESULT_DIR, INT8_DIR, TS_DIR, PTQ_RESULT_DIR]:
    print("\n", folder)
    for path in sorted(folder.glob("*")):
        print(" -", path.name)
